In [7]:
import os
import json
import random
from datetime import datetime, timedelta

os.makedirs("data", exist_ok=True)

stores = ['Warszawa', 'Krakow', 'Gdansk', 'Wroclaw']
categories = ['elektronika', 'odziez', 'zywnosc', 'ksiazki']
start_time = datetime(2023, 10, 25, 8, 0, 0)

with open("data/transactions_10k.jsonl", "w") as f:
    for i in range(10000):
        tx_num = random.randint(1, 9999)
        tx_time = start_time + timedelta(seconds=random.randint(0, 10800))
        record = {
            'tx_id': f'TX{tx_num:04d}',
            'user_id': f'u{random.randint(1, 20):02d}',
            'amount': round(random.uniform(5.0, 5000.0), 2),
            'store': random.choice(stores),
            'category': random.choice(categories),
            'timestamp': tx_time.strftime("%Y-%m-%d %H:%M:%S")
        }
        f.write(json.dumps(record) + "\n")

print("Gotowe")

Gotowe


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col, window, count, avg, round as _round, desc

# 1. Inicjalizacja Sparka
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

# 2. Wczytanie pliku
df = spark.read.json("data/transactions_10k.jsonl")

# 3. Zmiana formatu daty
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print("-" * 50)


# PRACA DOMOWA

# Zadanie 1: Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
print("Zadanie 1: Godzina w Gdańsku z najniższą średnią kwotą:")
gdansk_lowest_avg = (
    df.filter(col("store") == "Gdansk") 
    .groupBy(window("timestamp", "1 hour")) 
    .agg(
        _round(avg("amount"), 2).alias("srednia_kwota_PLN")
    )
    .orderBy("srednia_kwota_PLN") 
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_kwota_PLN"
    )
)
gdansk_lowest_avg.show(1) 

print("-" * 50)

# Zadanie 2: Policz ile transakcji per kategoria było w oknie 09:00–09:30.
print("Zadanie 2: Transakcje per kategoria między 09:00 a 09:30:")
kategorie_0900_0930 = (
    df.filter((col("timestamp") >= "2023-10-25 09:00:00") & (col("timestamp") < "2023-10-25 09:30:00"))
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_transakcji")
    )
    .orderBy(desc("liczba_transakcji"))
)
kategorie_0900_0930.show()

print("-" * 50)

# Zadanie 3: Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji.
print("Zadanie 3: Ćwierćgodzina z największą liczbą transakcji:")
szczyt_15min = (
    df.groupBy(window("timestamp", "15 minutes")) 
    .agg(
        count("tx_id").alias("liczba_transakcji")
    )
    .orderBy(desc("liczba_transakcji")) 
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_transakcji"
    )
)
szczyt_15min.show(1) 

Spark 4.0.0-preview2 — gotowy
--------------------------------------------------
Zadanie 1: Godzina w Gdańsku z najniższą średnią kwotą:
+-------------------+-------------------+-----------------+
|                 od|                 do|srednia_kwota_PLN|
+-------------------+-------------------+-----------------+
|2023-10-25 08:00:00|2023-10-25 09:00:00|          2487.19|
+-------------------+-------------------+-----------------+
only showing top 1 row

--------------------------------------------------
Zadanie 2: Transakcje per kategoria między 09:00 a 09:30:
+-----------+-----------------+
|   category|liczba_transakcji|
+-----------+-----------------+
|    zywnosc|              441|
|    ksiazki|              427|
|     odziez|              408|
|elektronika|              400|
+-----------+-----------------+

--------------------------------------------------
Zadanie 3: Ćwierćgodzina z największą liczbą transakcji:
+-------------------+-------------------+-----------------+
|    